# ARKENSTONE MASTER GPU v5 — clean one-click campaign

Use **Runtime → Change runtime type → T4 GPU**, then click **Run all**.

This notebook does not contain the long training loops. It pulls the exact audited runner commit, compiles it and its dependencies, runs a GPU smoke test, and only then launches the full campaign. This avoids the indentation/XLA/cache failures from the older notebooks.

Campaign order: **ARK-007R fresh retention replication → ARK-009 fact-set-disjoint non-arithmetic transfer → ARK-010 recovery after collapse**.


In [ ]:
import pathlib, shutil, subprocess, sys, torch

assert torch.cuda.is_available(), 'No CUDA GPU detected. Runtime -> Change runtime type -> T4 GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

PINNED_RUNNER_COMMIT = '6e544f8c0f14a2871c8a724f4790f6ee8c79ae15'
REPO_URL = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
repo = pathlib.Path('/content/An-Ra-the-new-AGI-v5')

# Always start clean so an old Colab clone or manual edit cannot leak into the run.
if repo.exists():
    shutil.rmtree(repo)

subprocess.run(['git', 'clone', '--depth', '20', '--branch', 'Arkenstone', REPO_URL, str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', PINNED_RUNNER_COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == PINNED_RUNNER_COMMIT, (actual, PINNED_RUNNER_COMMIT)
print('Pinned commit:', actual)

runner = repo / 'experiments/COLAB/run_master_v5.py'
deps = [
    runner,
    repo / 'experiments/ARK-001/run_ark001.py',
    repo / 'experiments/lib/ark_tasks.py',
]
for path in deps:
    subprocess.run([sys.executable, '-m', 'py_compile', str(path)], check=True)
print('Syntax/import dependency compile PASS')

# Runtime smoke test: manifest hash, GPU forward/backward, snapshot reload,
# continuation-order determinism, binding split invariants, one-step binding path.
subprocess.run([sys.executable, '-u', str(runner), '--smoke-test', '--budget-minutes', '240'], check=True)
print('GPU SMOKE TEST PASS — safe to start full campaign')


In [ ]:
import pathlib, subprocess, sys
runner = pathlib.Path('/content/An-Ra-the-new-AGI-v5/experiments/COLAB/run_master_v5.py')
subprocess.run([sys.executable, '-u', str(runner), '--budget-minutes', '240'], check=True)


In [ ]:
from pathlib import Path
results = Path('/content/arkenstone_v5_results')
print('Results directory:', results)
for p in sorted(results.glob('*')):
    print(p.name, p.stat().st_size, 'bytes')
print('If the automatic download popup was blocked, download ARKENSTONE_V5_RESULTS.zip from the Files pane.')
